# BEER McStas powder data reduction

- Audience: BEER users reducing McStas pulse-shaping powder data
- Prerequisites: Basic knowledge of Scipp and Sciline workflows

This guide reduces a McStas silicon-in-vanadium-can sample run using matching vanadium and empty-can runs. The result is `EmptyCanSubtractedIofDspacing`, the normalized intensity as a function of $d$-spacing after subtracting the empty can.

The example files are downloaded through the ESSdiffraction data registry. To run the notebook you will have to install `pooch` to access them: `pip install pooch`.

In [1]:
%matplotlib ipympl
import scipp as sc

from ess import beer, powder
import ess.beer.data  # noqa: F401
from ess.beer.types import DetectorBank
from ess.diffraction.peaks import dspacing_peaks_from_cif
from ess.powder.types import *

## Configure the workflow

Start with the BEER powder McStas workflow and set the run files. We use monitor-histogram normalization here because the McStas files include a wavelength monitor.

In [2]:
workflow = beer.BeerPowderMcStasWorkflowAnalytical(
    run_norm=powder.RunNormalization.monitor_histogram
)

workflow[Filename[SampleRun]] = beer.data.mcstas_powder_silicon_in_vanadium_can()
workflow[Filename[VanadiumRun]] = beer.data.mcstas_powder_vanadium()
workflow[Filename[EmptyCanRun]] = beer.data.mcstas_powder_empty_can()

Next set the detector bank and the $d$-spacing range. The notebook only reduces the north bank and only histograms the range from 0.6 Å to 2.0 Å.

In [3]:
workflow[DetectorBank] = DetectorBank.north
workflow[DspacingBins] = sc.linspace("dspacing", 0.6, 2.0, 701, unit="angstrom")

To keep the events in the output, to allow post reduction filtering or other processing, set `KeepEvents` to `True`.

In [4]:
workflow[KeepEvents[SampleRun]] = KeepEvents(True)
workflow[KeepEvents[VanadiumRun]] = KeepEvents(True)
workflow[KeepEvents[EmptyCanRun]] = KeepEvents(True)

workflow[MaskedDetectorIDs] = MaskedDetectorIDs({})
workflow[UncertaintyBroadcastMode] = UncertaintyBroadcastMode.drop

A finer region of interest can be applied with the standard powder masks. Mask callables return `True` for values to remove. This example keeps a broad two-theta range for the selected detector data and leaves the other masks disabled.

In [5]:
two_theta_min = sc.scalar(76.0, unit="deg").to(unit="rad")
two_theta_max = sc.scalar(104.0, unit="deg").to(unit="rad")

workflow[TwoThetaMask] = lambda two_theta: (two_theta < two_theta_min) | (
    two_theta > two_theta_max
)
workflow[TofMask] = None
workflow[WavelengthMask] = None

Some events can have ambiguous wavelength assignments. We first keep all detector events so the baseline spectrum shows the full result.

In [6]:
workflow[LookupTableRelativeErrorThreshold] = {
    "detector": float("inf"),
    "monitor_bunker": float("inf"),
    "monitor_cave": float("inf"),
}

## Compute the empty-can-subtracted spectrum

In [7]:
iof_dspacing = workflow.compute(EmptyCanSubtractedIofDspacing)
histogram = iof_dspacing.hist()
histogram

<scipp.DataArray>
Dimensions: Sizes[dspacing:700, ]
Coordinates:
* chopper_position          vector3              [m]  ()  (0.0061936, 0, 6.91241)
* detector_position         vector3              [m]  ()  (-2.10423, 0, 158.006)
* dspacing                  float64             [Å]  (dspacing [bin-edge])  [0.6, 0.602, ..., 1.998, 2]
* frame_cutoff_time         float64              [s]  ()  0.0508206
* mode                       string        <no unit>  ()  "4"
* nominal_time_at_chopper   float64              [s]  ()  0.00526935
  sample_position           vector3              [m]  ()  (-0.104235, 0, 158.003)
  source_position           vector3              [m]  ()  (-0.332522, 0, 6.91254)
* source_to_wavelength_definition_chopper_distance  float64              [m]  ()  6.91241
Data:
                            float64  [dimensionless]  (dspacing)  [0, 0, ..., 0.072723, 0.000832225]  [0, 0, ..., 0.00409623, 6.92599e-07]
Masks:
  zero_vanadium                bool        <no unit>  (dspacing)  [True, True, ..., False, False]

Compute the expected silicon peak positions from the CIF and draw the peaks that fall in the plotted $d$ range.

In [8]:
silicon_peaks = dspacing_peaks_from_cif("codid::1526655").coords["dspacing"]
silicon_peaks = silicon_peaks[
    (silicon_peaks > histogram.coords["dspacing"].min())
    & (silicon_peaks < histogram.coords["dspacing"].max())
]

fig = histogram.plot(
    title="Intensity over dspacing",
    xlabel=r"$d$-spacing [$\AA$]",
    ylabel=f"Intensity [{histogram.unit}]",
    xmin=sc.scalar(0.6, unit="angstrom"),
    xmax=sc.scalar(2.0, unit="angstrom"),
)

ymin, ymax = fig.ax.get_ylim()
for peak in silicon_peaks.values:
    fig.ax.axvline(peak, color="black", alpha=0.25, linewidth=1)

fig.ax.set_ylim(ymin, ymax)
fig

Querying the Crystallography Open Database for entry 1526655


Attempting to load CIF data with gemmi
Self-consistency of structure was verified by spglib


/home/runner/work/ess/ess/.pixi/envs/docs-essdiffraction/lib/python3.11/site-packages/NCrystal/cifutils.py:1566: NCrystalUserWarning: SG-227 available in multiple choices ("F d -3 m:1", "F d -3 m:2") and which one was not encoded explicitly in the _space_group_name_H-M_alt CIF field. Consider overriding the space group explicitly when loading this file.
  _nc_common.warn(f'SG-{sg.number} available in multiple'


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

## Reduce frame-overlap artifacts

The detector wavelength-assignment relative-error threshold removes events in regions where the wavelength assignment is ambiguous. The thresholds are set separately for the detector and the monitors.

In [9]:
thresholded_workflow = workflow.copy()
thresholded_workflow[LookupTableRelativeErrorThreshold] = {
    "detector": 0.005,
    "monitor_bunker": float("inf"),
    "monitor_cave": float("inf"),
}

thresholded_histogram = thresholded_workflow.compute(
    EmptyCanSubtractedIofDspacing
).hist()

The threshold removes events in specific regions of scattering angle and wavelength. We can see this directly by comparing `CorrectedDetector` before and after applying the threshold.

In [10]:
unfiltered_detector = workflow.compute(CorrectedDetector[SampleRun])
filtered_detector = thresholded_workflow.compute(CorrectedDetector[SampleRun])

two_theta_bins = sc.linspace("two_theta", 76.0, 104.0, 201, unit="deg").to(
    unit="rad"
)
wavelength_bins = sc.linspace("wavelength", 0.8, 3.2, 241, unit="angstrom")

unfiltered_map = unfiltered_detector.hist(
    two_theta=two_theta_bins,
    wavelength=wavelength_bins,
    dim=unfiltered_detector.dims,
)
filtered_map = filtered_detector.hist(
    two_theta=two_theta_bins,
    wavelength=wavelength_bins,
    dim=filtered_detector.dims,
)

In [11]:
filtered_map.plot(title="Kept by threshold", norm="log")

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [12]:
(unfiltered_map - filtered_map).plot(title="Removed by threshold", norm="log")

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [13]:
fig = sc.plot(
    {
        "baseline": histogram,
        "detector relative error < 0.02": thresholded_histogram,
    },
    xlabel=r"$d$-spacing [$\AA$]",
    ylabel=f"Intensity [{histogram.unit}]",
    xmin=sc.scalar(0.6, unit="angstrom"),
    xmax=sc.scalar(2.0, unit="angstrom"),
)

ymin, ymax = fig.ax.get_ylim()
for peak in silicon_peaks.values:
    fig.ax.axvline(peak, color="black", alpha=0.25, linewidth=1)

fig.ax.set_ylim(ymin, ymax)
fig

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…